# Convert between Transverse Mercator and WGS84

The Python code was rewritten from the C++ [PROJ](https://github.com/OSGeo/PROJ/tree/master) library.

## Structure implementations

In [1]:
from __future__ import annotations
from enum import Enum
import math

# Carto object implementations
class Coord:
    def __init__(self: Coord,
                 hemisphere: Hemisphere,
                 x: float,
                 y: float,
                 z: float,
                 zone: int) -> None:
        self.hemisphere = hemisphere
        self.x = x
        self.y = y
        self.z = z
        self.zone = zone

class EllipsoidDefinition:
    def __init__(self: EllipsoidDefinition,
                 semi_major: float,
                 inverse_flattening: float) -> None:
        self.a = semi_major
        self.rf = inverse_flattening
        self.f = 1 / self.rf
        self.b = self.a * (1 - self.f)
        self.e1_square = self.f * (2 - self.f)  # First eccentricity

    @property
    def e2_square(self: EllipsoidDefinition) -> float:
        '''
        The second eccentricity squared.
        '''
        return self.e1_square / (1 - self.e1_square)
    
    @property
    def e3_square(self: EllipsoidDefinition) -> float:
        '''
        The third eccentricity squared.
        '''
        return self.e1_square / (2 - self.e1_square)

class HelmertTransform:
    x = 0
    y = 0
    z = 0
    rx = 0
    ry = 0
    rz = 0
    s = 0
    len = 0
    
    def __init__(self: HelmertTransform,
                 params: list) -> None:
        if len(params) == 0:
            return
        elif len(params) == 3:
            self.x = params[0]
            self.y = params[1]
            self.z = params[2]
            self.len = 3
        elif len(params) == 7:
            self.x = params[0]
            self.y = params[1]
            self.z = params[2]
            self.rx = params[3]
            self.ry = params[4]
            self.rz = params[5]
            self.s = params[6]
            self.len = 7
        else:
            raise ValueError('The length should be 0, 3 or 7.')
        
    def __len__(self: HelmertTransform) -> int:
        return self.len

class Hemisphere(Enum):
    NORTH, SOUTH = range(2)

class ProjectionDefinition:
    def __init__(self: ProjectionDefinition,
                 ellipsoid: EllipsoidDefinition,
                 origin: double2,
                 shift: double2,
                 scale_factor: float,
                 transform: HelmertTransform) -> None:
        self.ellipsoid = ellipsoid
        self.origin = origin
        self.shift = shift
        self.scale_factor = scale_factor
        self.transform = transform

# Unity structure implementation
class double2:
    def __init__(self: double2,
                 x: float,
                 y: float) -> None:
        self.x = float(x)
        self.y = float(y)

    def __add__(self: double2,
                other: double2) -> double2:
        return double2(self.x + other.x, self.y + other.y)
    
    def __sub__(self: double2,
                other: double2) -> double2:
        return double2(self.x - other.x, self.y - other.y)
    
    def __mul__(self: double2,
                other: double2) -> double2:
        return double2(self.x * other.x, self.y * other.y)
    
    def __truediv__(self: double2,
                    other: double2) -> double2:
        return double2(self.x / other.x, self.y / other.y)
    
    def __pos__(self: double2) -> double2:
        return double2(self.x, self.y)

    def __neg__(self: double2) -> double2:
        return double2(-self.x, -self.y)
    
    def __repr__(self: double2) -> str:
        return f'double2({self.x}, {self.y})'

## DatumUtils class

In [ ]:
# Implemented structures:
# [Carto.Geodata] Coord, EllipsoidDefinition, HelmertTransform, Hemisphere & ProjectionDefinition
# [Unity.Mathematics] double2

class DatumUtils(object):
    @classmethod
    def clenshaw_summation(a: list[float],
                           sin_arg_r: float,
                           cos_arg_r: float,
                           sinh_arg_i: float,
                           cosh_arg_i: float) -> tuple[float, float]:
        '''
        The equivalent of `tmerc.cpp - clenS().` Note that this function returns `tuple[float, float]`, not original's `double`.
        '''
        size = len(a)
        r = 2 * cos_arg_r * cosh_arg_i
        i = -2 * sin_arg_r * sinh_arg_i

        hi1 = hr1 = hi = 0.0
        p = size - 1
        hr = a[p]
        p -= 1

        while p >= 0:
            hr2 = hr1
            hi2 = hi1
            hr1 = hr
            hi1 = hi
            hr = -hr2 + r * hr1 - i * hi1 + a[p]
            hi = -hi2 + i * hr1 + r * hi1
            p -= 1

        r_final = sin_arg_r * cosh_arg_i
        i_final = cos_arg_r * sinh_arg_i
        R = r_final * hr - i_final * hi
        I = r_final * hi + i_final * hr
        return R, I

## Transform class

In [ ]:
class Transform(object):
    @classmethod
    def transverse_mercator_to_wgs84():
        return None

## Test the methods